In [ ]:
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
import getFamaFrenchFactors as gff

In [ ]:
df =pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/df_dates.pkl')

In [ ]:
ff3_monthly = gff.famaFrench3Factor(frequency='m')#famaFrench3Factor(frequency='m')
ff3_monthly.rename(columns={"date_ff_factors": 'Date'}, inplace=True)
ff3_monthly.set_index('Date', inplace=True)

In [ ]:
ff3_monthly

In [ ]:
stock_returns = df['VIX'].resample('M').last().pct_change().dropna()
stock_returns.name = "Month_Rtn"
ff_data = ff3_monthly.merge(stock_returns,on='Date')

In [ ]:
ff_data.index = pd.to_datetime(ff_data.index, format = '%m/%d/%Y').strftime('%Y%m%d')

In [ ]:
ff_data.to_csv("./CSV/FamaFrenchVIX.csv")

In [ ]:
X = ff_data[['Mkt-RF', 'SMB', 'HML']]
y = ff_data['Month_Rtn'] - ff_data['RF']
X = sm.add_constant(X)
ff_model = sm.OLS(y, X).fit()
print(ff_model.summary())
intercept, b1, b2, b3 = ff_model.params

In [ ]:
rf = ff_data['RF'].mean()
market_premium = ff3_monthly['Mkt-RF'].mean()
size_premium = ff3_monthly['SMB'].mean()
value_premium = ff3_monthly['HML'].mean()

expected_monthly_return = rf + b1 * market_premium + b2 * size_premium + b3 * value_premium 
expected_yearly_return = expected_monthly_return * 12
print("Expected yearly return: " + str(expected_yearly_return))